In [13]:
!pip install -q -U transformers accelerate bitsandbytes datasets evaluate bert-score rouge-score sacrebleu textstat tqdm sacremoses

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.7 MB/s eta 0:00:00


In [1]:
import os
import gc
import time
import torch
import pandas as pd
import numpy as np

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

import evaluate
import textstat

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [2]:
# ==========================================
# LOAD MICROSOFT PHI-4-MINI-INSTRUCT
# ==========================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "microsoft/Phi-4-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("✅ Model loaded successfully!")
print("Device:", model.device)

Loading tokenizer...


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ Model loaded successfully!
Device: cuda:0


In [3]:
# ==========================================
# LOAD ASSET — 2359 SAMPLES
# ==========================================

asset_validation = load_dataset(
    "facebook/asset",
    split="validation"
)

asset_test = load_dataset(
    "facebook/asset",
    split="test"
)

# Combine validation + test
asset_df = pd.concat(
    [
        asset_validation.to_pandas(),
        asset_test.to_pandas()
    ],
    ignore_index=True
)

print("Validation samples:", len(asset_validation))
print("Test samples      :", len(asset_test))
print("Total samples     :", len(asset_df))

# Safety check
assert len(asset_df) == 2359, "Expected exactly 2359 samples!"

print("\n✅ ASSET dataset ready!")
print("Columns:", asset_df.columns.tolist())

print("\nExample:")
print(asset_df.iloc[0])

Validation samples: 2000
Test samples      : 359
Total samples     : 2359

✅ ASSET dataset ready!
Columns: ['original', 'simplifications']

Example:
original           Adjacent counties are Marin (to the south), Me...
simplifications    [countries next to it are Marin, Mendocino, La...
Name: 0, dtype: object


In [4]:
# ==========================================
# PHI ZERO-SHOT PROMPT
# ==========================================

MAX_NEW_TOKENS = 128
CHECKPOINT_EVERY = 25

def zero_shot_prompt(sentence):

    return f"""Simplify the following sentences using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Now simplify this sentence:

Original: {sentence}
Simplified:"""

In [5]:
# ==========================================
# LOAD EVALUATION METRICS
# ==========================================

print("Loading metrics...")

sari_metric = evaluate.load("sari")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ SARI loaded")
print("✅ ROUGE loaded")
print("✅ BERTScore loaded")

Loading metrics...
✅ SARI loaded
✅ ROUGE loaded
✅ BERTScore loaded


In [10]:
# ==========================================
# LOAD EVALUATION METRICS
# SAME PIPELINE AS GEMMA / QWEN
# ==========================================

import evaluate
import textstat
import numpy as np
import pandas as pd
import gc
import os
import time
import torch

print("Loading metrics...")

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu") # Added bleu_metric
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ SARI loaded")
print("✅ ROUGE loaded")
print("✅ BERTScore loaded")
print("✅ FKGL available through textstat")
print("✅ BLEU loaded") # Added confirmation for BLEU

Loading metrics...
✅ SARI loaded
✅ ROUGE loaded
✅ BERTScore loaded
✅ FKGL available through textstat
✅ BLEU loaded


In [7]:
def evaluate_one_sample(
    source,
    prediction,
    references
):

    prediction = str(prediction).strip()

    if not prediction:
        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }

    # SARI
    sari_result = sari_metric.compute(
        sources=[source],
        predictions=[prediction],
        references=[references]
    )

    sari = sari_result["sari"]

    # BLEU
    try:
        bleu_result = bleu_metric.compute(
            predictions=[prediction],
            references=[references],
            smooth=True
        )

        bleu = bleu_result["bleu"]

    except (ZeroDivisionError, ValueError):

        bleu = 0.0

    # ROUGE-L
    rouge_scores = []

    for reference in references:

        rouge_result = rouge_metric.compute(
            predictions=[prediction],
            references=[reference]
        )

        rouge_scores.append(
            rouge_result["rougeL"]
        )

    rouge_l = float(
        np.mean(rouge_scores)
    )

    # BERTScore
    bert_result = bertscore_metric.compute(
        predictions=[prediction] * len(references),
        references=references,
        lang="en"
    )

    bertscore_f1 = float(
        np.mean(bert_result["f1"])
    )

    # FKGL
    try:

        fkgl = textstat.flesch_kincaid_grade(
            prediction
        )

    except Exception:

        fkgl = 0.0

    return {
        "sari": float(sari),
        "bleu": float(bleu),
        "rougeL": rouge_l,
        "bertscore_f1": bertscore_f1,
        "fkgl": float(fkgl)
    }

print("✅ evaluate_one_sample() ready")

✅ evaluate_one_sample() ready


In [8]:
# ==========================================
# PHI — SINGLE SAMPLE TEST
# SAME AS GEMMA / QWEN
# ==========================================

example = asset_df.iloc[0]

source = example["original"]

prompt = zero_shot_prompt(source)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


# ==========================================
# RESET VRAM MEASUREMENT
# ==========================================

if torch.cuda.is_available():

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


# ==========================================
# GENERATION
# ==========================================

start_time = time.perf_counter()

with torch.no_grad():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

inference_time = (
    end_time - start_time
)


# ==========================================
# REMOVE PROMPT TOKENS
# ==========================================

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()


# ==========================================
# PEAK VRAM
# ==========================================

if torch.cuda.is_available():

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

else:

    peak_vram = None


# ==========================================
# DISPLAY
# ==========================================

print("SOURCE:")
print(source)

print("\nPHI:")
print(prediction)

print(
    "\nInference time:",
    round(inference_time, 3),
    "seconds"
)

print(
    "Peak VRAM:",
    round(peak_vram, 3),
    "GB"
)

SOURCE:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

PHI:
Marin, Mendocino, Lake, Napa, Solano, and Contra Costa are nearby counties.

Inference time: 4.35 seconds
Peak VRAM: 2.768 GB


In [11]:
test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():

    print(
        f"{metric}: {value:.4f}"
    )

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



METRICS
sari: 44.1069
bleu: 0.8432
rougeL: 0.6248
bertscore_f1: 0.9423
fkgl: 12.3000


In [13]:
# ==========================================
# PHI ZERO-SHOT — FULL ASSET EVALUATION
# FIXED: PRINT GENERATED TEXT
# SAME PIPELINE AS GEMMA / QWEN
# ==========================================

OUTPUT_FILE = "PHI_ZERO_SHOT_ASSET_2359_FINAL.csv"

CHECKPOINT_EVERY = 25

print("Output:", OUTPUT_FILE)


# ==========================================
# RESUME / NEW RUN
# ==========================================

if os.path.exists(OUTPUT_FILE):

    results_df = pd.read_csv(
        OUTPUT_FILE
    )

    completed_ids = set(
        results_df["sample_id"]
        .astype(int)
    )

    print(
        f"🔄 Existing results found: "
        f"{len(completed_ids)} completed."
    )

else:

    results_df = pd.DataFrame()

    completed_ids = set()

    print(
        "🆕 Starting from sample 1."
    )


# ==========================================
# MAIN EVALUATION LOOP
# ==========================================

for i in range(len(asset_df)):

    # =====================================
    # RESUME
    # =====================================

    if i in completed_ids:
        continue


    # =====================================
    # DATA
    # =====================================

    example = asset_df.iloc[i]

    source = example["original"]

    references = example["simplifications"]


    # =====================================
    # PROMPT
    # =====================================

    prompt = zero_shot_prompt(
        source
    )


    # =====================================
    # TOKENIZATION
    # =====================================

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    # =====================================
    # GPU MEMORY RESET
    # =====================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # =====================================
    # GENERATION
    # =====================================

    start_time = time.perf_counter()

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.perf_counter()

    inference_time = (
        end_time - start_time
    )


    # =====================================
    # DECODE
    # =====================================

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


    # =====================================
    # PEAK VRAM
    # IMPORTANT:
    # measured immediately after generation
    # before BERTScore
    # =====================================

    if torch.cuda.is_available():

        peak_vram = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )

    else:

        peak_vram = None


    # =====================================
    # METRICS
    # =====================================

    scores = evaluate_one_sample(
        source,
        prediction,
        references
    )


    # =====================================
    # RESULT
    # =====================================

    result = {

        "sample_id": i,

        "model": "Phi-4-mini-instruct",

        "prompting": "zero-shot",

        "original": source,

        "generated": prediction,

        "references": " ||| ".join(
            references
        ),

        "sari": scores["sari"],

        "bleu": scores["bleu"],

        "rougeL": scores["rougeL"],

        "bertscore_f1": scores["bertscore_f1"],

        "fkgl": scores["fkgl"],

        "inference_time_sec": inference_time,

        "peak_vram_gb": peak_vram
    }


    # =====================================
    # APPEND
    # =====================================

    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    completed_ids.add(i)


    # =====================================
    # CHECKPOINT
    # =====================================

    if (
        len(completed_ids) % CHECKPOINT_EVERY == 0
        or i == len(asset_df) - 1
    ):

        results_df.to_csv(
            OUTPUT_FILE,
            index=False
        )

        print(
            f"\n💾 CHECKPOINT: "
            f"{len(completed_ids)}/2359 saved"
        )


    # =====================================
    # DISPLAY ORIGINAL + GENERATED TEXT
    # =====================================

    print("\n" + "=" * 90)

    print(
        f"SAMPLE {i + 1}/2359"
    )

    print("\nORIGINAL:")
    print(source)

    print("\nPHI GENERATED:")
    print(prediction)

    print("\nMETRICS:")
    print(
        f"SARI          : {scores['sari']:.4f}"
    )

    print(
        f"BLEU          : {scores['bleu']:.4f}"
    )

    print(
        f"ROUGE-L       : {scores['rougeL']:.4f}"
    )

    print(
        f"BERTScore F1  : {scores['bertscore_f1']:.4f}"
    )

    print(
        f"FKGL          : {scores['fkgl']:.4f}"
    )

    print(
        f"Inference     : {inference_time:.3f} sec"
    )

    print(
        f"Peak VRAM     : {peak_vram:.3f} GB"
    )

    print("=" * 90)


    # =====================================
    # CLEANUP
    # =====================================

    del inputs
    del output_ids
    del generated_ids

    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

Streaming output truncated to the last 5000 lines.
SAMPLE 2098/2359

ORIGINAL:
As of 2000, the population was 89,148.

PHI GENERATED:
In 2000, the population was 89,148.

METRICS:
SARI          : 48.5570
BLEU          : 1.0000
ROUGE-L       : 0.7296
BERTScore F1  : 0.9768
FKGL          : 4.4500
Inference     : 0.885 sec
Peak VRAM     : 3.721 GB

SAMPLE 2099/2359

ORIGINAL:
Aliteracy (sometimes spelled alliteracy) is the state of being able to read but being uninterested in doing so.

PHI GENERATED:
Aliteracy means knowing how to read but not wanting to read.

METRICS:
SARI          : 47.2535
BLEU          : 0.7170
ROUGE-L       : 0.4379
BERTScore F1  : 0.9475
FKGL          : 5.8636
Inference     : 0.985 sec
Peak VRAM     : 3.772 GB

💾 CHECKPOINT: 2100/2359 saved

SAMPLE 2100/2359

ORIGINAL:
Mifepristone is a synthetic steroid compound used as a pharmaceutical.

PHI GENERATED:
Mifepristone is a drug made from synthetic steroids.

METRICS:
SARI          : 31.4049
BLEU          : 0.3357
R

In [15]:
import pandas as pd

# Load your CSV
df = pd.read_csv("PHI_ZERO_SHOT_ASSET_2359_FINAL.csv")

metrics = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]

summary = pd.DataFrame({
    "Min": df[metrics].min(),
    "Max": df[metrics].max(),
    "Mean": df[metrics].mean(),
    "Median": df[metrics].median(),
    "Average": df[metrics].mean()
})

print("=" * 90)
print("PHI — OVERALL STATISTICS")
print("=" * 90)
print(summary.round(4))

PHI — OVERALL STATISTICS
                       Min      Max     Mean   Median  Average
sari                0.0000  80.2756  39.7683  39.3723  39.7683
bleu                0.0000   1.0000   0.6192   0.6332   0.6192
rougeL              0.0000   0.9560   0.5412   0.5492   0.5412
bertscore_f1        0.0000   0.9939   0.9437   0.9468   0.9437
fkgl               -2.6200  24.8900   7.9426   7.7733   7.9426
inference_time_sec  0.1516   3.2965   1.2420   1.1641   1.2420
peak_vram_gb        3.7195   3.7804   3.7518   3.7722   3.7518


In [16]:
for col in ["sari", "bleu", "rougeL", "bertscore_f1", "fkgl"]:
    print(f"{col}: {(df[col] == 0).sum()} zeros")

sari: 3 zeros
bleu: 3 zeros
rougeL: 3 zeros
bertscore_f1: 3 zeros
fkgl: 3 zeros


In [17]:
zero_rows = df[df["bertscore_f1"] == 0]

print(zero_rows[[
    "original",
    "generated",
    "bertscore_f1"
]].head(20))

                                               original generated  \
1177  After a version of a software package has rema...       NaN   
1621  Normally a pawn moves by advancing a single sq...       NaN   
2307  That is because real estate, businesses and ot...       NaN   

      bertscore_f1  
1177           0.0  
1621           0.0  
2307           0.0  


In [18]:
valid_df = df[df["generated"].notna() & (df["generated"].astype(str).str.strip() != "")].copy()

metrics = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]

summary = pd.DataFrame({
    "Min": valid_df[metrics].min(),
    "Max": valid_df[metrics].max(),
    "Mean": valid_df[metrics].mean(),
    "Median": valid_df[metrics].median()
})

print("=" * 90)
print("PHI — VALID OUTPUT STATISTICS")
print("=" * 90)
print(summary.round(4))

print("\nTotal samples     :", len(df))
print("Valid generations :", len(valid_df))
print("Failed generations:", len(df) - len(valid_df))
print("Success rate      :", f"{len(valid_df)/len(df)*100:.3f}%")

PHI — VALID OUTPUT STATISTICS
                        Min      Max     Mean   Median
sari                13.0592  80.2756  39.8190  39.4065
bleu                 0.0538   1.0000   0.6200   0.6333
rougeL               0.0826   0.9560   0.5419   0.5494
bertscore_f1         0.8500   0.9939   0.9449   0.9469
fkgl                -2.6200  24.8900   7.9528   7.7733
inference_time_sec   0.3304   3.2965   1.2434   1.1644
peak_vram_gb         3.7195   3.7804   3.7518   3.7722

Total samples     : 2359
Valid generations : 2356
Failed generations: 3
Success rate      : 99.873%
